# Global snowmelt runoff onset icechunk store creation (v10+)

Creates and initializes the **icechunk** repository holding the Zarr v3 global output store that per-tile-per-water-year processing jobs commit into. Replaces `create_zarr_store.ipynb` (which pre-allocated the plain Zarr v2 store used through v9 -- that store stays frozen as published).

## Store design

- **Same five variables as v9** (`runoff_onset`, `runoff_onset_median`, `runoff_onset_mad`, `temporal_resolution`, `temporal_resolution_median`), int16 on disk, -9999 nodata, 0.1-day scaling for MAD/temporal resolution.
- **Zarr v3 sharding**: shards of (1 water_year, 2048, 2048) = exactly one tile x one water year, so each processing commit writes whole shards and concurrent tile jobs never touch the same object. Inner chunks of (1, 256, 256) keep point/station reads small (256 vs 512 benchmarked on Azure 2026-07: point-read p90 24% better at 256, everything else within noise).
- **Metadata-only init**: the template is lazy dask; only Zarr metadata + coordinates are written (`compute=False`, `write_empty_chunks=False`). Unprocessed regions simply have no chunks.
- **Manifest splitting** (persisted via `repo.save_config()`): one manifest per water year per array, so each of the ~50k processing commits rewrites only a small manifest instead of the full chunk-reference list. See [icechunk performance guide](https://icechunk.io/en/latest/guides/performance/).
- **Status = commit history**: processing state is derived from structured commit metadata (`global_snowmelt_runoff_onset/status.py`); there is no CSV/status file to maintain.


In [1]:
import icechunk
import xarray as xr

from global_snowmelt_runoff_onset.config import Config
from global_snowmelt_runoff_onset import store
from global_snowmelt_runoff_onset.provenance import collect_provenance

config = Config('config/global_config_v10.txt')
print(f'output repo: {config.global_runoff_icechunk_azure_prefix}')
print(f'shard: (1, {config.spatial_chunk_dim_zarr_output}, {config.spatial_chunk_dim_zarr_output}), '
      f'inner chunks: (1, {config.inner_chunk_dim}, {config.inner_chunk_dim})')

SAS token is valid until 2026-08-26 19:52 UTC (554.3 hours)
----------------------------------------
Configuration loaded:
config_name = global_config_v10
version = v10
resolution = 0.00072000072000072
bands = vv
mountain_snow_only = False
spatial_chunk_dim_s1_read = 2048
spatial_chunk_dim_s1_process = 512
spatial_chunk_dim_zarr_output = 2048
bbox_left = -179.999
bbox_right = 179.999
bbox_top = 84.048
bbox_bottom = -63.4074
expected_grid_shape = 204800, 499998
expected_tile_grid = 100, 245
wy_start = 2015
wy_end = 2025
trailing_buffer_days = 120
low_backscatter_threshold = 0.001
max_allowed_days_gap_per_orbit = 30
min_years_for_median_std = 3
extend_search_window_beyond_sdd_days = 16
min_consec_snow_days_for_seasonal_snow = 56
valid_tiles_geojson_path = processing/tile_data/global_tiles_with_seasonal_snow_v10.geojson
global_runoff_icechunk_azure_prefix = snowmelt/snowmelt_runoff_onset/global_runoff_onset_v10
inner_chunk_dim = 256
snow_phenology_zarr_store_azure_path = snowmelt/modis_sn

## Preview the template

Lazy dataset -- nothing is computed or uploaded here.


In [2]:
template_ds, encoding = store.build_template(config)
display(template_ds)
encoding['runoff_onset']

/home/eric/repos/global_snowmelt_runoff_onset/global_snowmelt_runoff_onset/store.py:72: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  template_ds = xr.combine_by_coords([


<xarray.Dataset> Size: 5TB
Dimensions:                     (water_year: 11, latitude: 204800,
                                 longitude: 499998)
Coordinates:
  * water_year                  (water_year) int64 88B 2015 2016 ... 2024 2025
  * latitude                    (latitude) float64 2MB 84.05 84.05 ... -63.41
  * longitude                   (longitude) float64 4MB -180.0 -180.0 ... 180.0
    spatial_ref                 int32 4B 4326
Data variables:
    runoff_onset                (water_year, latitude, longitude) int16 2TB dask.array<chunksize=(11, 2048, 2048), meta=np.ndarray>
    runoff_onset_median         (latitude, longitude) int16 205GB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    runoff_onset_mad            (latitude, longitude) int16 205GB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
    temporal_resolution         (water_year, latitude, longitude) int16 2TB dask.array<chunksize=(11, 2048, 2048), meta=np.ndarray>
    temporal_resolution_median  (latitude, longitude) int16 205GB dask.array<chunksize=(2048, 2048), meta=np.ndarray>
Attributes:
    title:           Global snowmelt runoff onset from Sentinel-1 SAR
    config_version:  v10

{'shards': (1, 2048, 2048),
 'chunks': (1, 256, 256),
 'compressors': [BloscCodec(_tunable_attrs={'typesize', 'shuffle'}, typesize=1, cname=<BloscCname.zstd: 'zstd'>, clevel=5, shuffle=<BloscShuffle.bitshuffle: 'bitshuffle'>, blocksize=0)],
 'dtype': 'int16',
 '_FillValue': np.int16(-9999),
 'fill_value': np.int16(-9999)}

## Create the repository and write the template

`Repository.create` fails if a repo already exists at the prefix -- it will not silently overwrite. To truly start over, delete the prefix first (deliberately manual):

```python
# import adlfs
# fs = adlfs.AzureBlobFileSystem(account_name=config.azure_storage_account, credential=config.sas_token)
# fs.rm(config.global_runoff_icechunk_azure_prefix, recursive=True)
```


In [3]:
import adlfs
fs = adlfs.AzureBlobFileSystem(account_name=config.azure_storage_account, credential=config.sas_token)
fs.rm(config.global_runoff_icechunk_azure_prefix, recursive=True)

In [4]:
repo = config.create_output_repo()  # persists manifest-splitting + retry config on-disk
snapshot_id = store.initialize_store(repo, config, extra_metadata={'provenance': collect_provenance()})
print(f'initialized: {snapshot_id}')

/home/eric/repos/global_snowmelt_runoff_onset/global_snowmelt_runoff_onset/store.py:72: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  template_ds = xr.combine_by_coords([


initialized: 79TDNF2985PQA4HYWN20


## Verify


In [5]:
session = repo.readonly_session('main')
global_zarr_ds = xr.open_zarr(session.store, zarr_format=3, consolidated=False, decode_coords='all')
display(global_zarr_ds)
for var in global_zarr_ds.data_vars:
    print(var, global_zarr_ds[var].encoding.get('shards'), global_zarr_ds[var].encoding.get('chunks'),
          global_zarr_ds[var].encoding.get('dtype'))

<xarray.Dataset> Size: 16TB
Dimensions:                     (latitude: 204800, longitude: 499998,
                                 water_year: 11)
Coordinates:
  * latitude                    (latitude) float64 2MB 84.05 84.05 ... -63.41
  * longitude                   (longitude) float64 4MB -180.0 -180.0 ... 180.0
  * water_year                  (water_year) int64 88B 2015 2016 ... 2024 2025
    spatial_ref                 int32 4B ...
Data variables:
    runoff_onset_median         (latitude, longitude) float32 410GB dask.array<chunksize=(256, 256), meta=np.ndarray>
    runoff_onset_mad            (latitude, longitude) float64 819GB dask.array<chunksize=(256, 256), meta=np.ndarray>
    temporal_resolution         (water_year, latitude, longitude) float64 9TB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    runoff_onset                (water_year, latitude, longitude) float32 5TB dask.array<chunksize=(1, 256, 256), meta=np.ndarray>
    temporal_resolution_median  (latitude, longitude) float64 819GB dask.array<chunksize=(256, 256), meta=np.ndarray>
Attributes:
    title:           Global snowmelt runoff onset from Sentinel-1 SAR
    config_version:  v10

runoff_onset_median (2048, 2048) (256, 256) int16
runoff_onset_mad (2048, 2048) (256, 256) int16
temporal_resolution (1, 2048, 2048) (1, 256, 256) int16
runoff_onset (1, 2048, 2048) (1, 256, 256) int16
temporal_resolution_median (2048, 2048) (256, 256) int16


## Notes

- **Appending a future water year** (e.g. WY2026): update `WY_end` in a new config, `to_zarr(..., append_dim='water_year')` a metadata-only slice (or resize via zarr), then dispatch tile jobs with `--water-years 2026`; composite commits refresh per-tile as those jobs run (staleness is tracked automatically).
- **At publication**: `repo.expire_snapshots(...)` + `repo.garbage_collect(...)` to compact the ~50k processing commits, then `repo.create_tag('v10.0', snapshot_id=...)` so readers can pin the released version.
